# Citi Bike Data Engineering Project


## Project Overview
Design and implement a normalised PostgreSQL database that integrates 2016 Citi Bike trip data with Newark Airport weather data, complete with a robust ETL pipeline and analytical views using DataFlow-Pro.

## Environment Setup

In [1]:
# Install required packages (run once)
# pip install dataflow-pro psycopg[binary] python-dotenv sqlalchemy

import dataflow_pro as dfp
from pathlib import Path
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Database configuration
DB_CONFIG = {
    'host': os.getenv('DB_HOST', 'localhost'),
    'port': os.getenv('DB_PORT', '5432'),
    'database': os.getenv('DB_NAME', 'citibike_db'),
    'user': os.getenv('DB_USER', 'postgres'),
    'password': os.getenv('DB_PASSWORD', 'your_password')
}

# Create connection string
CONNECTION_STRING = f"postgresql+psycopg://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"

# Create report directory
REPORT_DIR = Path("reports")
REPORT_DIR.mkdir(exist_ok=True)

print("✓ Environment setup complete!")
print(f"✓ Database: {DB_CONFIG['database']}")
print(f"✓ Host: {DB_CONFIG['host']}:{DB_CONFIG['port']}")

Data loading error occurred


✓ Environment setup complete!
✓ Database: citibike_db
✓ Host: localhost:5432


## Task 1: Data Preparation - Load All Data


In [2]:
from pathlib import Path
import dataflow_pro as dfp
import pyarrow as pa
from dataflow_pro.core.engine_selector import DataFrameWrapper

# Define data directory - corrected path
data_dir = Path('data-sources/data')

# Load all 12 Citi Bike CSV files
print("Loading Citi Bike trip data...")
citibike_files = [data_dir / f'JC-2016{i:02d}-citibike-tripdata.csv' for i in range(1, 13)]

# Load all files
trips_data_list = [dfp.load(f) for f in citibike_files]

# Extract arrow tables and concatenate
arrow_tables = [df.arrow_table for df in trips_data_list]
combined_arrow = pa.concat_tables(arrow_tables)

# Wrap back into DataFrameWrapper
trips_data = DataFrameWrapper(combined_arrow)

print(f"✓ Loaded {len(trips_data.to_pandas()):,} total trips")

# Load weather data
weather_data = dfp.load(data_dir / 'newark_airport_2016.csv')
print(f"✓ Loaded {len(weather_data.to_pandas()):,} weather records")

# Store datasets for later use
datasets = {
    'citibike_trips': trips_data,
    'weather': weather_data
}

print("\n✓ Data loading complete!")

Loading Citi Bike trip data...
✓ Loaded 247,584 total trips
✓ Loaded 366 weather records

✓ Data loading complete!


## Task 2: Data Quality Assessment
### Generate Data Quality Assessment Report

In [3]:
from pathlib import Path
import dataflow_pro as dfp
from dataflow_pro.eda.reporter import ReportConfig

# Assuming REPORT_DIR is defined somewhere
REPORT_DIR = Path('reports')  # Adjust as needed
REPORT_DIR.mkdir(parents=True, exist_ok=True)

# Generate comprehensive quality assessment using built-in functions
print("\nGenerating data quality assessments...")

# Assess Citi Bike trips
trips_quality = dfp.assess_data_quality(trips_data)
print(f"✓ Trips Quality Score: {trips_quality.overall_score:.1f}%")

# Assess weather data
weather_quality = dfp.assess_data_quality(weather_data)
print(f"✓ Weather Quality Score: {weather_quality.overall_score:.1f}%")

# Create report configuration with quality metrics enabled
report_config = ReportConfig(
    include_profiling=True,
    include_quality=True,
    include_recommendations=True,
    include_data_sample=True,
    sample_rows=10
)

# Generate combined HTML report
dfp.eda.generate_stacked_report(
    datasets,
    output_path=REPORT_DIR / 'citibike_data_quality_assessment.html',
    title='Citi Bike & Weather Data Quality Assessment',
    config=report_config
)

print(f"\n✓ Quality assessment report: {REPORT_DIR / 'citibike_data_quality_assessment.html'}")


Generating data quality assessments...
✓ Trips Quality Score: 0.0%
✓ Weather Quality Score: 0.0%

✓ Quality assessment report: reports/citibike_data_quality_assessment.html


## Task 3: Business Rules Validation

In [4]:
from dataflow_pro.rules import RuleGenerator, RuleValidator, ValidationReportIntegrator

# Define custom business rules (simplified specification)
business_rules = [
    {
        'table': 'citibike_trips',
        'column': 'Trip Duration',
        'rule_type': 'numeric_range',
        'parameters': {'min': 60, 'max': 86400},
        'description': 'Trip duration between 1 minute and 24 hours'
    },
    {
        'table': 'citibike_trips',
        'column': 'Start Station Latitude',
        'rule_type': 'numeric_range',
        'parameters': {'min': 40.0, 'max': 41.0},
        'description': 'Latitude within NJ range'
    },
    {
        'table': 'citibike_trips',
        'column': 'User Type',
        'rule_type': 'categorical_enum',
        'parameters': {'allowed_values': ['Customer', 'Subscriber']},
        'description': 'Valid user types only'
    },
    {
        'table': 'citibike_trips',
        'column': 'Birth Year',
        'rule_type': 'numeric_range',
        'parameters': {'min': 1940, 'max': 2000},
        'description': 'Reasonable birth years'
    }
]

# Generate and validate rules
print("Validating business rules...")
rule_generator = RuleGenerator()
all_rules = rule_generator.create_business_rules(business_rules)

validator = RuleValidator()
validation_summary = validator.validate_all(datasets, all_rules)

print(f"✓ Validation complete!")
print(f"  - Rules passed: {validation_summary.passed_rules}/{validation_summary.total_rules}")
print(f"  - Quality score: {validation_summary.data_quality_score:.1f}%")
print(f"  - Total violations: {validation_summary.total_violations:,}")

# Generate validation report
integrator = ValidationReportIntegrator()
report_html = integrator.generate_validation_html_section(validation_summary)

with open(REPORT_DIR / 'citibike_rule_validation_report.html', 'w', encoding='utf-8') as f:
    f.write(report_html)

print(f"\n✓ Validation report: {REPORT_DIR / 'citibike_rule_validation_report.html'}")

Validating business rules...
✓ Validation complete!
  - Rules passed: 3/4
  - Quality score: 75.0%
  - Total violations: 380

✓ Validation report: reports/citibike_rule_validation_report.html


### Review the rows that failed the rule validations

In [5]:
import pandas as pd

# Get the trips dataframe
trips_df = datasets['citibike_trips'].to_pandas()

# Filter rows where User Type is NOT 'Customer' or 'Subscriber'
allowed_values = ['Customer', 'Subscriber']
invalid_user_type_rows = trips_df[~trips_df['User Type'].isin(allowed_values)]

print(f"Total rows with invalid User Type: {len(invalid_user_type_rows):,}")
print(f"\nUnique invalid values: {invalid_user_type_rows['User Type'].unique()}")

# Display the rows
invalid_user_type_rows

Total rows with invalid User Type: 380

Unique invalid values: ['']
Categories (3, object): ['Subscriber', 'Customer', '']


,Trip Duration,Start Time,Stop Time,Start Station ID,Start Station Name,Start Station Latitude,Start Station Longitude,End Station ID,End Station Name,End Station Latitude,End Station Longitude,Bike ID,User Type,Birth Year,Gender
25267,156,2016-03-23 09:08:34,2016-03-23 09:11:11,3214,Essex Light Rail,40.712772,-74.036484,3183,Exchange Place,40.716248,-74.033463,24444,,1987.0,1
25668,164,2016-03-23 22:17:45,2016-03-23 22:20:29,3183,Exchange Place,40.716248,-74.033463,3214,Essex Light Rail,40.712772,-74.036484,24675,,1987.0,1
25894,171,2016-03-24 11:46:39,2016-03-24 11:49:31,3214,Essex Light Rail,40.712772,-74.036484,3183,Exchange Place,40.716248,-74.033463,24697,,1987.0,1
26189,204,2016-03-24 20:45:45,2016-03-24 20:49:10,3183,Exchange Place,40.716248,-74.033463,3214,Essex Light Rail,40.712772,-74.036484,24387,,1987.0,1
26630,380,2016-03-25 19:15:56,2016-03-25 19:22:17,3183,Exchange Place,40.716248,-74.033463,3184,Paulus Hook,40.714146,-74.033554,24412,,1987.0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
245137,1266,2016-12-24 18:21:54,2016-12-24 18:43:00,3186,Grove St PATH,40.719585,-74.043114,3199,Newport Pkwy,40.728745,-74.032104,26200,,1991.0,1
246572,1791,2016-12-28 18:51:00,2016-12-28 19:20:52,3202,Newport PATH,40.727222,-74.033760,3199,Newport Pkwy,40.728745,-74.032104,26194,,1982.0,1
246573,1248,2016-12-28 18:51:07,2016-12-28 19:11:55,3202,Newport PATH,40.727222,-74.033760,3199,Newport Pkwy,40.728745,-74.032104,26292,,1987.0,2
246623,1130,2016-12-28 20:52:18,2016-12-28 21:11:08,3199,Newport Pkwy,40.728745,-74.032104,3199,Newport Pkwy,40.728745,-74.032104,26194,,1982.0,1


## Task 4: Data Transformation

In [6]:
import pandas as pd
import pyarrow as pa
from dataflow_pro.core import DataFrameWrapper

# Convert to pandas for transformation
df_trips = datasets['citibike_trips'].to_pandas()
df_weather = datasets['weather'].to_pandas()

# 1. Users Dimension
users_df = df_trips[['User Type', 'Birth Year', 'Gender']].drop_duplicates().reset_index(drop=True)
users_df.insert(0, 'User ID', range(1, len(users_df) + 1))

# 2. Bike Stations Dimension
start_stations = df_trips[['Start Station ID', 'Start Station Name',
                           'Start Station Latitude', 'Start Station Longitude']]
start_stations.columns = ['Station ID', 'Station Name', 'Latitude', 'Longitude']
end_stations = df_trips[['End Station ID', 'End Station Name',
                         'End Station Latitude', 'End Station Longitude']]
end_stations.columns = ['Station ID', 'Station Name', 'Latitude', 'Longitude']
bike_stations_df = pd.concat([start_stations, end_stations]).drop_duplicates('Station ID').reset_index(drop=True)

# 3. Weather Stations Dimension
weather_stations_df = df_weather[['STATION', 'NAME']].drop_duplicates().reset_index(drop=True)
weather_stations_df.columns = ['Station', 'Name']
weather_stations_df.insert(0, 'Weather Station ID', range(1, len(weather_stations_df) + 1))

# 4. Trips Fact Table
trips_fact_df = df_trips.merge(users_df, on=['User Type', 'Birth Year', 'Gender'], how='left')
trips_fact_df = trips_fact_df[['User ID', 'Start Station ID', 'End Station ID',
                                'Trip Duration', 'Start Time', 'Stop Time', 'Bike ID']]

# 5. Weather Dates Fact Table
weather_dates_df = df_weather.merge(weather_stations_df, left_on=['STATION', 'NAME'],
                                    right_on=['Station', 'Name'], how='left')
weather_dates_df = weather_dates_df.drop(columns=['STATION', 'NAME', 'Station', 'Name'])
cols = ['Weather Station ID'] + [c for c in weather_dates_df.columns if c != 'Weather Station ID']
weather_dates_df = weather_dates_df[cols]

# Convert to DataFrameWrapper for DataFlow-Pro
transformed_datasets = {
    'trips': DataFrameWrapper(pa.Table.from_pandas(trips_fact_df)),
    'users': DataFrameWrapper(pa.Table.from_pandas(users_df)),
    'bike_stations': DataFrameWrapper(pa.Table.from_pandas(bike_stations_df)),
    'weather_stations': DataFrameWrapper(pa.Table.from_pandas(weather_stations_df)),
    'weather_dates': DataFrameWrapper(pa.Table.from_pandas(weather_dates_df))
}

print(f"✓ Transformed 2 datasets into 5 tables:")
print(f"  • Trips: {len(trips_fact_df):,} rows")
print(f"  • Users: {len(users_df):,} rows")
print(f"  • Bike Stations: {len(bike_stations_df):,} rows")
print(f"  • Weather Stations: {len(weather_stations_df):,} rows")
print(f"  • Weather Dates: {len(weather_dates_df):,} rows")

✓ Transformed 2 datasets into 5 tables:
  • Trips: 247,584 rows
  • Users: 210 rows
  • Bike Stations: 102 rows
  • Weather Stations: 1 rows
  • Weather Dates: 366 rows


## Task 5: Schema Design

In [7]:
from dataflow_pro.schema import design_schema_definition, save_er_diagram_as_png
from dataflow_pro.schema.schema_reporter import EnhancedSchemaReporter, SchemaReportConfig

# Design star schema from transformed datasets
print("\nDesigning star schema...")

# Generate schema with validation disabled to avoid the bug
schema_definition = design_schema_definition(
    datasets=transformed_datasets,
    schema_name='citibike_star_schema',
    enable_pattern_analysis=True,
    enable_model_validation=False,  # Disabled to bypass the bug
    business_context={
        'fact_tables': ['trips', 'weather_dates'],
        'dimension_tables': ['users', 'bike_stations', 'weather_stations'],
        'business_objectives': [
            'daily_trip_analysis',
            'weather_impact_on_ridership',
            'station_popularity',
            'user_demographics'
        ]
    }
)

print(f"✓ Star schema designed with {len(schema_definition.tables)} tables")

# Generate schema report configuration
config = SchemaReportConfig(
    include_er_diagram=True,
    include_table_analysis=True,
    include_relationship_analysis=True,
    include_recommendations=True,
    include_pattern_analysis=True,
    include_validation_results=False,  # Disabled to avoid the bug
    include_ddl_generation=True
)

# Generate the comprehensive report
reporter = EnhancedSchemaReporter()
schema_design, report_path = reporter.generate_comprehensive_schema_report(
    datasets=transformed_datasets,
    schema_name='citibike_star_schema',
    output_path=str(REPORT_DIR / 'schema_design_report.html'),
    title='Citi Bike Star Schema Design',
    report_config=config
)

print(f"✓ Schema report: {report_path}")

# Generate ER diagram
try:
    diagram_path = save_er_diagram_as_png(
        schema_design,
        output_path=REPORT_DIR / 'er_diagram.png',
        title='Citi Bike Star Schema'
    )
    print(f"✓ ER diagram: {diagram_path}")
except ImportError:
    print("⚠️  matplotlib not available - ER diagram skipped")


Designing star schema...


/Users/harrylane/Documents/Portfolio/Data Engineering/citibike-data-engineering/citibike-data-engineering/venv/lib/python3.9/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/harrylane/Documents/Portfolio/Data Engineering/citibike-data-engineering/citibike-data-engineering/venv/lib/python3.9/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/Users/harrylane/Documents/Portfolio/Data Engineering/citibike-data-engineering/citibike-data-engineering/venv/lib/python3.9/site-packages/numpy/lib/function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/Users/harrylane/Documents/Portfolio/Data Engineering/citibike-data-engineering/citibike-data-engineering/venv/lib/python3.9/site-packages/numpy/lib/function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


✓ Star schema designed with 5 tables


/Users/harrylane/Documents/Portfolio/Data Engineering/citibike-data-engineering/citibike-data-engineering/venv/lib/python3.9/site-packages/dataflow_pro/schema/diagram_generator.py:384: UserWarning: Glyph 10067 (\N{BLACK QUESTION MARK ORNAMENT}) missing from font(s) DejaVu Sans.
  plt.tight_layout()
/Users/harrylane/Documents/Portfolio/Data Engineering/citibike-data-engineering/citibike-data-engineering/venv/lib/python3.9/site-packages/dataflow_pro/schema/diagram_generator.py:2094: UserWarning: Glyph 10067 (\N{BLACK QUESTION MARK ORNAMENT}) missing from font(s) DejaVu Sans.
  fig.savefig(buffer, format="png", bbox_inches="tight", facecolor="white", edgecolor="none", dpi=150)


✓ Schema report: reports/schema_design_report.html


/Users/harrylane/Documents/Portfolio/Data Engineering/citibike-data-engineering/citibike-data-engineering/venv/lib/python3.9/site-packages/dataflow_pro/schema/diagram_generator.py:2040: UserWarning: Glyph 10067 (\N{BLACK QUESTION MARK ORNAMENT}) missing from font(s) DejaVu Sans.
  fig.savefig(output_path, dpi=dpi, bbox_inches="tight", facecolor="white", edgecolor="none")


✓ ER diagram: reports/er_diagram.png


## Task 6: Generate PostgreSQL DDL

In [9]:
from dataflow_pro.schema.model import generate_postgresql_ddl

# Generate SQL DDL statements
print("\nGenerating PostgreSQL DDL...")

ddl_statements = generate_postgresql_ddl(schema_definition)

# Save DDL to SQL file
sql_file = Path('sql') / 'create_star_schema.sql'
sql_file.parent.mkdir(exist_ok=True)

with open(sql_file, 'w') as f:
    f.write("-- Citi Bike Star Schema DDL\n")
    f.write("-- Auto-generated by DataFlow-Pro\n\n")
    f.write(ddl_statements)

print(f"✓ DDL generated: {sql_file}")
print(f"  - Tables: {len(schema_definition.tables)}")
print(f"  - Relationships: {len(schema_definition.relationships)}")


Generating PostgreSQL DDL...
✓ DDL generated: sql/create_star_schema.sql
  - Tables: 5
  - Relationships: 4


## Task 7: Create Database Tables

In [15]:
from sqlalchemy import create_engine, text

# Execute DDL to create tables
print("\nCreating database tables...")

engine = create_engine(CONNECTION_STRING)

# Create corrected DDL with proper column names
corrected_ddl = '''
-- Drop tables if they exist
DROP TABLE IF EXISTS trips CASCADE;
DROP TABLE IF EXISTS users CASCADE;
DROP TABLE IF EXISTS bike_stations CASCADE;
DROP TABLE IF EXISTS weather_stations CASCADE;
DROP TABLE IF EXISTS weather_dates CASCADE;

-- Create tables with quoted column names
CREATE TABLE users (
    "User ID" SMALLINT NOT NULL,
    "User Type" TEXT NOT NULL,
    "Birth Year" REAL NOT NULL,
    "Gender" SMALLINT NOT NULL,
    PRIMARY KEY ("User ID")
);

CREATE TABLE bike_stations (
    "Station ID" SMALLINT NOT NULL,
    "Station Name" TEXT NOT NULL,
    "Latitude" REAL NOT NULL,
    "Longitude" REAL NOT NULL,
    PRIMARY KEY ("Station ID")
);

CREATE TABLE weather_stations (
    "Weather Station ID" SMALLINT NOT NULL,
    "Station" TEXT NOT NULL,
    "Name" TEXT NOT NULL,
    PRIMARY KEY ("Weather Station ID")
);

CREATE TABLE weather_dates (
    "Weather Station ID" SMALLINT NOT NULL,
    "DATE" DATE NOT NULL,
    "AWND" REAL NOT NULL,
    "PGTM" TEXT,
    "PRCP" REAL NOT NULL,
    "SNOW" REAL NOT NULL,
    "SNWD" REAL NOT NULL,
    "TAVG" SMALLINT NOT NULL,
    "TMAX" SMALLINT NOT NULL,
    "TMIN" SMALLINT NOT NULL,
    "TSUN" TEXT,
    "WDF2" SMALLINT NOT NULL,
    "WDF5" REAL NOT NULL,
    "WSF2" REAL NOT NULL,
    "WSF5" REAL NOT NULL,
    PRIMARY KEY ("Weather Station ID", "DATE")
);

CREATE TABLE trips (
    "User ID" SMALLINT NOT NULL,
    "Start Station ID" SMALLINT NOT NULL,
    "End Station ID" SMALLINT NOT NULL,
    "Trip Duration" INTEGER NOT NULL,
    "Start Time" TEXT NOT NULL,
    "Stop Time" TEXT NOT NULL,
    "Bike ID" SMALLINT NOT NULL,
    PRIMARY KEY ("Start Time", "Stop Time"),
    FOREIGN KEY ("User ID") REFERENCES users("User ID"),
    FOREIGN KEY ("Start Station ID") REFERENCES bike_stations("Station ID"),
    FOREIGN KEY ("End Station ID") REFERENCES bike_stations("Station ID")
);

ALTER TABLE weather_dates
ADD CONSTRAINT fk_weather_dates_station
FOREIGN KEY ("Weather Station ID") REFERENCES weather_stations("Weather Station ID");
'''

with engine.connect() as conn:
    # Execute each statement
    for statement in corrected_ddl.split(';'):
        if statement.strip():
            conn.execute(text(statement))
    conn.commit()

print("✓ Database tables created successfully!")

# Verify tables
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT table_name
        FROM information_schema.tables
        WHERE table_schema = 'public'
        ORDER BY table_name
    """))
    tables = [row[0] for row in result]

print(f"\nCreated tables:")
for table in tables:
    print(f"  - {table}")


Creating database tables...
✓ Database tables created successfully!

Created tables:
  - bike_stations
  - trips
  - users
  - weather_dates
  - weather_stations


## Task 8: ETL Pipeline - Load Data into PostgreSQL

In [30]:
# Ultra-Minimal ETL Pipeline - Final Working Version
print("=" * 40)
print("LOADING DATA TO POSTGRESQL")
print("=" * 40)

with engine.connect() as conn:
    # Clear existing data
    for table in ['trips', 'weather_dates', 'users', 'bike_stations', 'weather_stations']:
        conn.execute(text(f'DELETE FROM {table}'))

    # Load each table with data cleaning
    for table_name in ['users', 'bike_stations', 'weather_stations', 'weather_dates', 'trips']:
        if table_name not in transformed_datasets:
            continue

        print(f"Loading {table_name}...")
        df = transformed_datasets[table_name].to_pandas()

        # Clean data based on table
        if table_name == 'users':
            df['Birth Year'] = df['Birth Year'].fillna(1970)
        elif table_name == 'weather_dates':
            # Fill NULL values with reasonable defaults for weather data
            numeric_cols = ['AWND', 'PRCP', 'SNOW', 'SNWD', 'TAVG', 'TMAX', 'TMIN', 'WDF2', 'WDF5', 'WSF2', 'WSF5']
            for col in numeric_cols:
                if col in df.columns:
                    df[col] = df[col].fillna(0)
        elif table_name == 'trips':
            df.columns = [col.strip('"') for col in df.columns]
            df = df.drop_duplicates(subset=['Start Time', 'Stop Time'], keep='first')

        # Insert data row by row (slower but handles all edge cases)
        for i, row in df.iterrows():
            try:
                if table_name == 'users':
                    conn.execute(text(
                        'INSERT INTO users ("User ID", "User Type", "Birth Year", "Gender") VALUES (:uid, :utype, :byear, :gender)'
                    ), {'uid': int(row['User ID']), 'utype': str(row['User Type'])[:50], 'byear': float(row['Birth Year']), 'gender': int(row['Gender'])})

                elif table_name == 'bike_stations':
                    conn.execute(text(
                        'INSERT INTO bike_stations ("Station ID", "Station Name", "Latitude", "Longitude") VALUES (:sid, :sname, :lat, :lon)'
                    ), {'sid': int(row['Station ID']), 'sname': str(row['Station Name'])[:100], 'lat': float(row['Latitude']), 'lon': float(row['Longitude'])})

                elif table_name == 'weather_stations':
                    conn.execute(text(
                        'INSERT INTO weather_stations ("Weather Station ID", "Station", "Name") VALUES (:wsid, :station, :name)'
                    ), {'wsid': int(row['Weather Station ID']), 'station': str(row['Station'])[:50], 'name': str(row['Name'])[:100]})

                elif table_name == 'weather_dates':
                    conn.execute(text('''
                        INSERT INTO weather_dates ("Weather Station ID", "DATE", "AWND", "PGTM", "PRCP",
                                                  "SNOW", "SNWD", "TAVG", "TMAX", "TMIN", "TSUN",
                                                  "WDF2", "WDF5", "WSF2", "WSF5")
                        VALUES (:wsid, :date, :awnd, :pgtm, :prcp, :snow, :snwd, :tavg, :tmax, :tmin,
                               :tsun, :wdf2, :wdf5, :wsf2, :wsf5)
                    '''), {
                        'wsid': int(row['Weather Station ID']), 'date': row['DATE'],
                        'awnd': float(row['AWND']), 'pgtm': row['PGTM'], 'prcp': float(row['PRCP']),
                        'snow': float(row['SNOW']), 'snwd': float(row['SNWD']), 'tavg': int(row['TAVG']),
                        'tmax': int(row['TMAX']), 'tmin': int(row['TMIN']), 'tsun': row['TSUN'],
                        'wdf2': int(row['WDF2']), 'wdf5': float(row['WDF5']), 'wsf2': float(row['WSF2']), 'wsf5': float(row['WSF5'])
                    })

                elif table_name == 'trips':
                    conn.execute(text('''
                        INSERT INTO trips ("User ID", "Start Station ID", "End Station ID", "Trip Duration",
                                         "Start Time", "Stop Time", "Bike ID")
                        VALUES (:uid, :ssid, :esid, :duration, :stime, :etime, :bid)
                    '''), {
                        'uid': int(row['User ID']), 'ssid': int(row['Start Station ID']),
                        'esid': int(row['End Station ID']), 'duration': int(row['Trip Duration']),
                        'stime': str(row['Start Time']), 'etime': str(row['Stop Time']),
                        'bid': int(row['Bike ID'])
                    })

            except Exception as e:
                if i < 5:  # Only log first few errors
                    print(f"  Warning: Skipped row {i}: {str(e)[:100]}...")
                continue

        conn.commit()

        # Count actual loaded records
        result = conn.execute(text(f'SELECT COUNT(*) FROM {table_name}'))
        count = result.scalar()
        print(f"  ✓ {table_name}: {count:,} records loaded")

print(f"\n✓ ETL Pipeline completed!")

# Final summary
with engine.connect() as conn:
    print("\nFinal table counts:")
    total = 0
    for table in ['users', 'bike_stations', 'weather_stations', 'weather_dates', 'trips']:
        result = conn.execute(text(f'SELECT COUNT(*) FROM {table}'))
        count = result.scalar()
        total += count
        print(f"  {table}: {count:,} rows")
    print(f"\nTotal records loaded: {total:,}")

print("\n✓ Data loading complete!")

LOADING DATA TO POSTGRESQL
Loading users...
  ✓ users: 210 records loaded
Loading bike_stations...
  ✓ bike_stations: 102 records loaded
Loading weather_stations...
  ✓ weather_stations: 1 records loaded
Loading weather_dates...
  ✓ weather_dates: 366 records loaded
Loading trips...
  ✓ weather_dates: 366 records loaded
Loading trips...
  ✓ trips: 247,496 records loaded

✓ ETL Pipeline completed!

Final table counts:
  users: 210 rows
  bike_stations: 102 rows
  weather_stations: 1 rows
  weather_dates: 366 rows
  trips: 247,496 rows

Total records loaded: 248,175

✓ Data loading complete!
  ✓ trips: 247,496 records loaded

✓ ETL Pipeline completed!

Final table counts:
  users: 210 rows
  bike_stations: 102 rows
  weather_stations: 1 rows
  weather_dates: 366 rows
  trips: 247,496 rows

Total records loaded: 248,175

✓ Data loading complete!


## Task 9: Create Analytical SQL Views

In [35]:
# Create a single comprehensive view using SQLGenerator and RelationshipDetector results
print("=" * 70)
print("CREATING UNIFIED ANALYTICAL VIEW")
print("=" * 70)

# Generate unified view using all relationships from schema design
unified_view_sql = """
CREATE OR REPLACE VIEW v_unified_citibike_analysis AS
SELECT
    -- Trip information
    t."Start Time" as trip_start_time,
    t."Stop Time" as trip_stop_time,
    t."Trip Duration" as trip_duration_seconds,
    t."Bike ID" as bike_id,

    -- User demographics
    u."User Type" as user_type,
    u."Birth Year" as birth_year,
    u."Gender" as gender,
    (2016 - u."Birth Year") as user_age_in_2016,

    -- Start station details
    bs_start."Station Name" as start_station_name,
    bs_start."Latitude" as start_latitude,
    bs_start."Longitude" as start_longitude,

    -- End station details
    bs_end."Station Name" as end_station_name,
    bs_end."Latitude" as end_latitude,
    bs_end."Longitude" as end_longitude,

    -- Weather information
    wd."DATE" as weather_date,
    wd."TMAX" as max_temp_f,
    wd."TMIN" as min_temp_f,
    wd."TAVG" as avg_temp_f,
    wd."PRCP" as precipitation_inches,
    wd."SNOW" as snow_inches,
    wd."AWND" as avg_wind_speed,
    ws."Name" as weather_station_name

FROM trips t
INNER JOIN users u ON t."User ID" = u."User ID"
INNER JOIN bike_stations bs_start ON t."Start Station ID" = bs_start."Station ID"
INNER JOIN bike_stations bs_end ON t."End Station ID" = bs_end."Station ID"
LEFT JOIN weather_dates wd ON DATE(t."Start Time") = wd."DATE"
LEFT JOIN weather_stations ws ON wd."Weather Station ID" = ws."Weather Station ID"
ORDER BY t."Start Time"
;
"""

# Create the unified view
with engine.connect() as conn:
    try:
        conn.execute(text(unified_view_sql))
        conn.commit()
        print("✓ Created unified view: v_unified_citibike_analysis")

        # Verify with sample data
        result = conn.execute(text("SELECT COUNT(*) FROM v_unified_citibike_analysis"))
        count = result.scalar()
        print(f"✓ View contains {count:,} records")

    except Exception as e:
        print(f"✗ Failed: {e}")

print("\n✓ Unified analytical view complete!")

CREATING UNIFIED ANALYTICAL VIEW
✓ Created unified view: v_unified_citibike_analysis
✓ View contains 247,496 records

✓ Unified analytical view complete!
✓ View contains 247,496 records

✓ Unified analytical view complete!
